# Test thực chiến: Upload PDF thật + Chat bằng model local (Ollama qwen3)

Chạy server trước: `python3 server.py` (cổng 8001), rồi Run All cells.
Test trên 2-3 PDF thật trong `src_update/input` + file của bạn.

In [ ]:
import requests, json

API = "http://127.0.0.1:8001"
print(requests.get(f"{API}/api/health", timeout=10).json())

In [ ]:
from pathlib import Path

# Đổi 3 file này thành PDF bạn muốn test
PDFS = [
    "/Users/nhan/Documents/00_Inbox/src_update/input/integration_formulas.pdf",
    "/Users/nhan/Documents/00_Inbox/src_update/input/TOAN-BO-CONG-THUC-TOAN-CAP-3-full-www.MATHVN.com_.pdf",
    "/Users/nhan/Documents/00_Inbox/Python - basic.pdf",
]
for p in PDFS:
    print(Path(p).name, "->", Path(p).exists(), f"{Path(p).stat().st_size/1e6:.1f}MB" if Path(p).exists() else "")

In [ ]:
def upload(pdf_path):
    with open(pdf_path, "rb") as f:
        r = requests.post(f"{API}/api/upload", files={"f": (Path(pdf_path).name, f, "application/pdf")}, timeout=300)
    r.raise_for_status()
    return r.json()

DOCS = {}
for p in PDFS:
    info = upload(p)
    DOCS[p] = info
    print(f"OK {Path(p).name}: {info['num_pages']} trang, {info['num_chunks']} chunks, doc_id={info['doc_id']}")

In [ ]:
def ask(doc_id, question, top_k=5):
    r = requests.post(f"{API}/api/chat", json={"doc_id": doc_id, "question": question, "top_k": top_k}, timeout=400)
    r.raise_for_status()
    return r.json()

QUESTIONS = [
    "Tóm tắt tài liệu này trong 5 gạch đầu dòng.",
    "Liệt kê các công thức quan trọng nhất kèm số trang.",
    "Tài liệu này hướng dẫn điều gì? Nêu các phần chính.",
]

In [ ]:
# Chat trên từng PDF đã upload
for p, info in DOCS.items():
    print("="*80)
    print("FILE:", Path(p).name)
    for q in QUESTIONS[:2]:
        res = ask(info["doc_id"], q)
        print(f"\nQ: {q}")
        print(f"A ({res['elapsed_s']}s, trích dẫn trang {res['citations']}):")
        print(res["answer"][:1500])
        print("-"*60)

## Nhận xét kết quả
- Đạt nếu: trả lời đúng nội dung file + có `[số trang]` + số trang bấm được trong `reader.html`.
- Chưa đạt nếu: trích sai trang -> tăng `top_k`, hoặc chunk nhỏ lại trong `server.py`.